### Сравнение моделей 

Сравним модели:

1) Ridge
2) CatBoost
3) Гибрид Ridge + CatBoost
4) LSTM


Напомню, baseline:

MAE=0.2149, RMSE=0.2711, MAPE=16.88%

## Подготовим данные

In [1]:
# Импорты для работы с данными, визуализацией и нейросетями
import math
import random
from dataclasses import dataclass

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

# Для нормализации данных и расчета метрик
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path

# PyTorch для построения и обучения нейросетей
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Определяем устройство: используем GPU если доступен, иначе CPU
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.mps.is_available() else "cpu"))
print("device:", device)

Device: mps


In [ ]:
data_dir = Path("../data/")
dataset = "g5_2xlarge_6h_after_2024-07_with_features.parquet"

df = pl.read_parquet(data_dir / dataset)



In [5]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

feature_cols = [c for c in df.columns if c not in ["datetime", "target_сost"]]
cat_cols = ["hour", "weekday", "month", "day", "week_of_year", "is_weekend", "is_holiday"]
num_cols = [c for c in feature_cols if c not in cat_cols]
num_idx = [feature_cols.index(c) for c in num_cols]

X = df[feature_cols].to_numpy()
y = df["target_сost"].to_numpy()

tscv = TimeSeriesSplit(n_splits=5)


### Ridge

In [6]:
for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X[train_idx].copy(), X[test_idx].copy()
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train[:, num_idx] = scaler.fit_transform(X_train[:, num_idx])
    X_test[:, num_idx] = scaler.transform(X_test[:, num_idx])

    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    print(f"Fold {fold+1}: MAE={mae:.4f}, RMSE={rmse:.4f}, MAPE={mape:.2f}%")

Fold 1: MAE=0.2875, RMSE=0.3331, MAPE=29.82%
Fold 2: MAE=0.1037, RMSE=0.1669, MAPE=9.75%
Fold 3: MAE=0.1096, RMSE=0.1749, MAPE=11.19%
Fold 4: MAE=0.1285, RMSE=0.1938, MAPE=12.64%
Fold 5: MAE=0.1039, RMSE=0.1516, MAPE=8.15%


### CatBoost

In [7]:
from catboost import CatBoostRegressor


for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train = df[feature_cols][train_idx]
    X_test = df[feature_cols][test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        cat_features=cat_cols,
        verbose=False
    )
    model.fit(X_train.to_pandas(), y_train)
    y_pred = model.predict(X_test.to_pandas())
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    print(f"Fold {fold+1}: MAE={mae:.4f}, RMSE={rmse:.4f}, MAPE={mape:.2f}%")

Fold 1: MAE=0.3251, RMSE=0.3446, MAPE=37.00%
Fold 2: MAE=0.1247, RMSE=0.1730, MAPE=12.34%
Fold 3: MAE=0.1469, RMSE=0.1955, MAPE=15.44%
Fold 4: MAE=0.1440, RMSE=0.1990, MAPE=14.34%
Fold 5: MAE=0.1722, RMSE=0.2572, MAPE=12.06%


#### MAE=0.1722, RMSE=0.2572, MAPE=12.06%

## Гибрид Ridge + CatBoost

In [ ]:
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor

cat_feature_idx = [feature_cols.index(c) for c in cat_cols if c in feature_cols]

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train = df[feature_cols][train_idx].to_pandas()
    X_test = df[feature_cols][test_idx].to_pandas()
    y_train, y_test = y[train_idx], y[test_idx]

    # шаг 1 — Ridge
    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train)
    
    ridge_pred_train = ridge.predict(X_train_scaled)
    ridge_pred_test = ridge.predict(X_test_scaled)

    # шаг 2 — CatBoost на остатках
    residuals_train = y_train - ridge_pred_train

    catboost = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        cat_features=cat_cols,
        devices="0",
        verbose=False
    )
    catboost.fit(X_train, residuals_train)
    residuals_pred = catboost.predict(X_test)

    # шаг 3 — суммируем
    y_pred = ridge_pred_test + residuals_pred

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    print(f"Fold {fold+1}: MAE={mae:.4f}, RMSE={rmse:.4f}, MAPE={mape:.2f}%")

Fold 1: MAE=0.3297, RMSE=0.3658, MAPE=36.21%
Fold 2: MAE=0.1121, RMSE=0.1720, MAPE=10.61%
Fold 3: MAE=0.1437, RMSE=0.1961, MAPE=15.10%
Fold 4: MAE=0.1473, RMSE=0.2004, MAPE=14.63%
Fold 5: MAE=0.1325, RMSE=0.1857, MAPE=9.73%


## LSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

# параметры
SEQ_LEN = 28
HORIZON = 4
BATCH_SIZE = 32
EPOCHS = 50

# нормализация
scaler = StandardScaler()
feature_cols_lstm = [c for c in df.columns if c not in ["datetime"]]
data = df[feature_cols_lstm].to_numpy().astype(np.float32)

# Dataset
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len, horizon, target_idx):
        self.data = data
        self.seq_len = seq_len
        self.horizon = horizon
        self.target_idx = target_idx

    def __len__(self):
        return len(self.data) - self.seq_len - self.horizon + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len + self.horizon - 1, self.target_idx]
        return torch.tensor(x), torch.tensor(y)

# LSTM модель
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

In [12]:
target_idx = feature_cols_lstm.index("cost")
split_idx = int(len(data) * 0.8)

train_data = scaler.fit_transform(data[:split_idx])
test_data = scaler.transform(data[split_idx:])

train_dataset = TimeSeriesDataset(train_data, SEQ_LEN, HORIZON, target_idx)
test_dataset = TimeSeriesDataset(test_data, SEQ_LEN, HORIZON, target_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = LSTMModel(input_size=data.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {train_loss/len(train_loader):.4f}")

# оценка
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        pred = model(X_batch).cpu().numpy()
        y_pred.extend(pred.reshape(-1))
        y_true.extend(y_batch.numpy().reshape(-1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# обратная нормализация
cost_std = scaler.scale_[target_idx]
cost_mean = scaler.mean_[target_idx]
y_true = y_true * cost_std + cost_mean
y_pred = y_pred * cost_std + cost_mean

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"LSTM: MAE={mae:.4f}, RMSE={rmse:.4f}, MAPE={mape:.2f}%")

Epoch 10/50, Loss: 0.0216
Epoch 20/50, Loss: 0.0156
Epoch 30/50, Loss: 0.0082
Epoch 40/50, Loss: 0.0070
Epoch 50/50, Loss: 0.0049
LSTM: MAE=0.0562, RMSE=0.0798, MAPE=4.04%


### LSTM: MAE=0.0562, RMSE=0.0798, MAPE=4.04%

# Итоги

| Модель | MAE | RMSE | MAPE |
|---|---|---|---|
| Baseline (lag_1w) | 0.2149 | 0.2711 | 16.88% |
| CatBoost | 0.1722 | 0.2572 | 12.06% |
| Ridge + CatBoost | 0.1325 | 0.1857 | 9.73% |
| Ridge | 0.1039 | 0.1516 | 8.15% |
| **LSTM** | **0.0562** | **0.0798** | **4.04%** |



Перейдем к тюнингу lstm